# Notebook 02 — Preprocessing
## STOP 4 — Train/Test Split

In [1]:
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import TensorDataset, DataLoader

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

df = fetch_california_housing(as_frame=True).frame.copy()
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

def split_stats(name, yv):
    print(name, "mean=", round(float(yv.mean()), 4), "std=", round(float(yv.std()), 4), "n=", len(yv))

split_stats("train", y_train)
split_stats("val", y_val)
split_stats("test", y_test)

train mean= 2.0682 std= 1.1527 n= 14035
val mean= 2.0931 std= 1.1761 n= 2477
test mean= 2.055 std= 1.1449 n= 4128


## STOP 5 — Feature Scaling

In [2]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

with open(PROC / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

np.savez(PROC / "dataset_arrays.npz",
         X_train=X_train_s.astype(np.float32),
         X_val=X_val_s.astype(np.float32),
         X_test=X_test_s.astype(np.float32),
         y_train=y_train.to_numpy(dtype=np.float32),
         y_val=y_val.to_numpy(dtype=np.float32),
         y_test=y_test.to_numpy(dtype=np.float32))
print("Saved scaler and arrays. y is intentionally not scaled.")

Saved scaler and arrays. y is intentionally not scaled.


## STOP 6 — Tensors & DataLoader

In [3]:
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_val_t = torch.tensor(X_val_s, dtype=torch.float32)
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train.to_numpy(dtype=np.float32)).unsqueeze(1)
y_val_t = torch.tensor(y_val.to_numpy(dtype=np.float32)).unsqueeze(1)
y_test_t = torch.tensor(y_test.to_numpy(dtype=np.float32)).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64, shuffle=False)

print("X_train:", X_train_t.shape, "y_train:", y_train_t.shape)
print("X_val:", X_val_t.shape, "y_val:", y_val_t.shape)
print("X_test:", X_test_t.shape, "y_test:", y_test_t.shape)

X_train: torch.Size([14035, 8]) y_train: torch.Size([14035, 1])
X_val: torch.Size([2477, 8]) y_val: torch.Size([2477, 1])
X_test: torch.Size([4128, 8]) y_test: torch.Size([4128, 1])
